In [ ]:
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, confusion_matrix
from PIL import Image
from tqdm import tqdm

In [ ]:
class VinDrMLODataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        
        # Mapeamento Binário (BI-RADS 1, 2, 3 = Benigno(0) | BI-RADS 4, 5 = Maligno(1))
        self.label_map = {1: 0, 2: 0, 3: 0, 4: 1, 5: 1}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Caminho do DICOM
        img_path = f"{self.root_dir}/{row['study_id']}/{row['image_id']}.dicom"
        
        # Leitura e Normalização do DICOM
        ds = pydicom.dcmread(img_path)
        pixel_array = ds.pixel_array.astype(float)
        
        # Normalização Min-Max para a imagem médica
        pixel_array = (pixel_array - np.min(pixel_array)) / (np.max(pixel_array) - np.min(pixel_array))
        pixel_array = (pixel_array * 255).astype(np.uint8)
        
        # Converte para imagem PIL em RGB (necessário para os pesos da ResNet)
        image = Image.fromarray(pixel_array).convert('RGB')
        
        # =========================================================
        # ALINHAMENTO DA MAMA (O Pulo do Gato)
        # =========================================================
        # Como confirmado, lateralidade é sempre 'L' ou 'R'
        lateralidade = row['laterality'] 
        
        # Espelha a mama direita para que todas fiquem orientadas como a esquerda
        if lateralidade == 'R':
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            
        # Pega a classe e converte para Binário
        label = self.label_map[row['breast_birads']]
        
        # Aplica Transformações (Tensor, Resize, Normalize...)
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# --- Transformações (Atenção: NÃO há RandomHorizontalFlip aqui!) ---
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomRotation(10), # Apenas rotação leve para augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Carregamento e Preparação do CSV ---
csv_path = "../dataset/VinDr-Mammo/breast-level_annotations.csv"
images_dir = "../dataset/vindr-mammo/images"

df_completo = pd.read_csv(csv_path)

# Filtra apenas MLO (Verifique se no CSV chama 'view' ou 'view_position')
df_mlo = df_completo[df_completo['view'] == 'MLO'].copy()

# Remove possíveis linhas sem BI-RADS anotado, se houver
df_mlo = df_mlo.dropna(subset=['breast_birads'])

# --- Split sem Data Leakage (por study_id) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df_mlo, groups=df_mlo['study_id']))

df_train = df_mlo.iloc[train_idx]
df_val = df_mlo.iloc[val_idx]

print(f"Total Imagens MLO: {len(df_mlo)} | Treino: {len(df_train)} | Validação: {len(df_val)}")

# --- DataLoaders ---
BATCH_SIZE = 16 # Ajuste para 8 ou 4 se tiver erro de falta de memória de vídeo (OOM)

train_dataset = VinDrMLODataset(dataframe=df_train, root_dir=images_dir, transform=train_transform)
val_dataset = VinDrMLODataset(dataframe=df_val, root_dir=images_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Modelo ResNet50 ---
model = models.resnet50(weights='IMAGENET1K_V1')

# Troca a última camada para 2 classes (Benigno vs Maligno)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

# --- Pesos das Classes para o Desbalanceamento ---
# Casos malignos são minoria. Damos um peso maior para a Classe 1 (Maligno)
# Exemplo: Peso 1.0 para Benigno e 8.0 para Maligno. (Ajuste se precisar de mais sensibilidade)
weights = torch.tensor([1.0, 8.0]).to(device) 

criterion = nn.CrossEntropyLoss(weight=weights)

# Optimizer com um Learning Rate baixo, ideal para transfer learning
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
num_epochs = 20
best_auc = 0.0

for epoch in range(num_epochs):
    print(f"\n--- Época {epoch+1}/{num_epochs} ---")
    
    # ==================================
    # TREINAMENTO
    # ==================================
    model.train()
    train_loss = 0.0
    
    loop_treino = tqdm(train_loader, desc="Treinamento", leave=False)
    
    for images, labels in loop_treino:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        loop_treino.set_postfix(loss=loss.item())
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # ==================================
    # VALIDAÇÃO
    # ==================================
    model.eval()
    val_loss = 0.0
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        loop_val = tqdm(val_loader, desc="Validação", leave=False)
        for images, labels in loop_val:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            # Probabilidade da classe 1 (Maligno)
            probs = F.softmax(outputs, dim=1)[:, 1]
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    val_loss = val_loss / len(val_loader.dataset)
    
    # ==================================
    # CÁLCULO DAS MÉTRICAS
    # ==================================
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    auc = roc_auc_score(all_labels, all_probs)
    
    print(f"Loss Treino: {train_loss:.4f} | Loss Validação: {val_loss:.4f}")
    print(f"Matriz de Confusão -> TP:{tp} | FN:{fn} | TN:{tn} | FP:{fp}")
    print(f"Sensibilidade (Recall): {sensitivity:.4f}")
    print(f"Especificidade:       {specificity:.4f}")
    print(f"AUC-ROC:              {auc:.4f}")
    
    # ==================================
    # SALVAR MELHOR MODELO
    # ==================================
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'melhor_modelo_vindr_mlo_binario.pth')
        print(f"🔥 Novo melhor modelo salvo! (AUC: {best_auc:.4f})")